In [1]:
import pandas as pd
import seaborn as sns
from utils import parse_har_files_to_dataframe
from common import BASE_HAR_DIR

sns.set_theme(style="whitegrid", context="paper")
pd.set_option('display.max_colwidth', 128)

df = parse_har_files_to_dataframe(BASE_HAR_DIR, test=False, use_cache=True)

# SST - Detection method
- We use requests initiated by Google Tag Manager script and find that they always contain the same 36 URL parameters
- We then search for these parameters in requests not initiaed by a script from googletagmanager.com
- (TODO) Analyzing the scripts that initiated these requests (containing the 36 params), we find that they are always initiated by a first-party GTM script. We detect first-party GTM scripts by analyzing their content.

In [2]:
df['param_names_str'] = df['param_names'].apply(lambda x: ', '.join(x))

In [ ]:
# list rows where the initiator domain contains googletagmanager.com
# params include gtm and the (request) entity is not Google
df_gtm = df[df['initiator_domains'].apply(lambda x: 'googletagmanager.com' in x)]
df_gtm = df_gtm[df_gtm.param_names_str.str.contains('gtm') & (df_gtm.entity != 'Google')]
df_gtm[['site_hostname', 'exp_mode', 'consent_mode', 'initiator_domains', 'req_hostname', 'param_names_str']]

In [ ]:
df_gtm.req_hostname.unique().tolist()

In [ ]:
df_gtm['param_names_str'].value_counts().head(5).to_frame()

In [6]:
from functools import reduce

sets = df_gtm['param_names'].map(set)

# Step 2: Find the intersection of all sets
gtm_intersection = reduce(set.intersection, sets)

# Display the intersection
print(len(gtm_intersection), sorted(list(gtm_intersection)))

36 ['_p', '_s', 'are', 'cid', 'dl', 'dma', 'dma_cps', 'dt', 'ecid', 'frm', 'gcd', 'gcs', 'gtm', 'npa', 'pscdl', 'sct', 'seg', 'sid', 'sr', 'sst.rnd', 'sst.tft', 'sst.ude', 'tag_exp', 'tfd', 'tid', 'uaa', 'uab', 'uafvl', 'uam', 'uamb', 'uap', 'uapv', 'uaw', 'ul', 'ur', 'v']


In [ ]:
df_ngtm = df[(df.param_names_str.str.contains('gtm')) &
             (df.param_names_str.str.contains('sr')) &
            #  (df.param_names_str.str.contains('dt')) &
            #  (df.param_names_str.str.contains('dl')) &
             (df.entity != 'Google')]
df_ngtm.req_hostname.value_counts().to_frame()

In [21]:
sets = df_ngtm['param_names'].map(set)

# Step 2: Find the intersection of all sets
ngtm_intersection = reduce(set.intersection, sets)

# Display the intersection
print(len(ngtm_intersection), sorted(list(ngtm_intersection)))

36 ['_p', '_s', 'are', 'cid', 'dl', 'dma', 'dma_cps', 'dt', 'ecid', 'frm', 'gcd', 'gcs', 'gtm', 'npa', 'pscdl', 'sct', 'seg', 'sid', 'sr', 'sst.rnd', 'sst.tft', 'sst.ude', 'tag_exp', 'tfd', 'tid', 'uaa', 'uab', 'uafvl', 'uam', 'uamb', 'uap', 'uapv', 'uaw', 'ul', 'ur', 'v']


In [ ]:
df_ngtm[df_ngtm['initiator_domains'].apply(lambda x: 'googletagmanager.com' not in x)].initiator_urls


In [9]:
ngtm_intersection == gtm_intersection

True

In [ ]:
expected_params = ngtm_intersection
GROUP_COLS = ['exp_mode', 'consent_mode', 'country', 'site_domain']

# ilter rows where 'param_names' contains all expected_params
df_all_gtm = df[df['param_names'].apply(lambda x: expected_params.issubset(x))]
df_all_gtm.drop_duplicates(['consent_mode', 'site_domain', 'req_hostname'])[
    ['exp_mode', 'consent_mode', 'country', 'site_domain', 'req_hostname']].\
        sort_values(['country', 'consent_mode', 'exp_mode'])

In [ ]:
sgtm_hosts = df_all_gtm.req_hostname.unique().tolist()
len(sgtm_hosts)

In [ ]:
import whois
import socket
for sgtm_host in sgtm_hosts:
    try:
        ip_address = socket.gethostbyname(sgtm_host)
        # print(whois.whois(ip_address))
        whois_org = whois.whois(ip_address).get('org', '')
        whois_org = whois_org.lower() if whois_org else ''
        whois_domains = whois.whois(ip_address).get('domain_name')
        if isinstance(whois_domains, list):
            whois_domains = list(set([domain.lower() for domain in whois_domains]))
            whois_domains = ' '.join(whois_domains).lower() if whois_domains else ''
        elif whois_domains:
            whois_domains = whois_domains.lower()
        else:
            whois_domains = ''
        # print("whois_domains", whois_domains, "whois_org", whois_org)
        # print(whois_org, whois_domains)
        if "google" in whois_org or "google" in whois_domains or "1e100.net" in whois_domains:
            print("Hosted on Google", sgtm_host, )
            pass
        else:
            print("Not hosted on Google", sgtm_host, ip_address, whois_org, whois_domains)
    except Exception as e:
        print(sgtm_host, e)

In [ ]:
df_all_gtm[~df_all_gtm.url.str.contains('collect?v', regex=False)][
    ['site_domain', 'consent_mode', 'exp_mode', 'req_hostname', 'url']].\
        req_hostname.value_counts().to_frame()

In [ ]:
df[
    df.param_names_str.str.contains('dl') &
    df.param_names_str.str.contains('dt') &
    df.param_names_str.str.contains('sr') &
    ~df.entity.isin(['Google']) &
    ~df['param_names'].apply(lambda x: expected_params.issubset(x))][
       ['site_domain', 'req_hostname', 'param_names_str', 'entity', 'initiator_domains']]

In [ ]:
df_gtm[~df_gtm.param_names_str.str.contains('richsst')]